# CS/INFO 5304 — Phase 1: Dataset Feasibility
**Team:** Zane Mroue (zsm7), Grace Myers (gm586), Samy Lokanandi (sl3539), Ali Hasan (ah2434)

---

## Research Questions

Our project looks at whether Yelp business/review data can tell us something about rent trends at the zip code level. Specifically our research questions are:

1. **Is there a correlation between the rhetoric of business category labels on Yelp, for example (“New American” or "Artisanal" ) and increased rent prices within the neighborhood?**

2. **Does the frequency of “gentrifiction terms” in Yelp reviews like “aesthetic” and “vibe” correlate strongly with future rent appreciation?**

The basic idea is that the cultural shifts you can see on Yelp might show up before or alongside economic shifts like rent going up.

## Raw Data Sources

**Yelp Open Dataset** ([link](https://business.yelp.com/data/resources/open-dataset/))  
Public dataset from Yelp for academic use. Has business metadata (location, categories, price range) and user reviews (text, date, stars). We mainly use `business.json` and `review.json`.

**Zillow Observed Rent Index (ZORI)** ([link](https://www.zillow.com/research/data/))  
Monthly rent index at the zip code level from Zillow Research. We're using the "ZORI (Smoothed): All Homes Plus Multifamily" time series.

Both are publicly available. no API keys or scraping needed.

---
## Data Loading & Cleaning

### Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import json
import os
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

ROOT = os.getcwd()
if os.path.basename(ROOT) == 'notebooks':
    ROOT = os.path.dirname(ROOT)

YELP_BUSINESS_PATH = os.path.join(ROOT, 'data', 'raw', 'yelp_academic_dataset_business.json')
YELP_REVIEW_PATH = os.path.join(ROOT, 'data', 'raw', 'yelp_academic_dataset_review.json')
ZILLOW_RENT_PATH = os.path.join(ROOT, 'data', 'raw', 'zillow_zori.csv')
OUTPUT_PATH = os.path.join(ROOT, 'data', 'processed', 'analysis_ready.csv')

### Load Yelp Business Data

Loading the full business dataset, then filtering down to just food/dining businesses since that's what our research questions focus on. We also extract the price range attribute and flag businesses whose categories match our gentrification-associated list (coffee shops, wine bars, etc...)

In [2]:
business_df = pd.read_json(YELP_BUSINESS_PATH, lines=True)
print(f"{len(business_df):,} total businesses")
print(f"Columns: {list(business_df.columns)}")
business_df.head(2)

150,346 total businesses
Columns: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."


In [3]:
business_df['categories'] = business_df['categories'].fillna('')

food_keywords = ['Restaurant', 'Food', 'Coffee', 'Cafe', 'Café', 'Bar', 
                 'Bakery', 'Diner', 'Bistro', 'Brewery', 'Pub', 'Pizza',
                 'Ice Cream', 'Juice', 'Deli', 'Bagel', 'Donut']

food_mask = business_df['categories'].apply(
    lambda x: any(kw.lower() in x.lower() for kw in food_keywords)
)
food_biz = business_df[food_mask].copy()
print(f"Food/dining businesses: {len(food_biz):,} ({len(food_biz)/len(business_df)*100:.1f}% of total)")

Food/dining businesses: 70,336 (46.8% of total)


In [4]:
#RestaurantsPriceRange2 is Yelp's $ to $$$$ scale (1-4)
def extract_price_range(attr):
    if isinstance(attr, dict):
        val = attr.get('RestaurantsPriceRange2')
        if val is not None:
            try:
                return int(val)
            except (ValueError, TypeError):
                return np.nan
    return np.nan

food_biz['price_range'] = food_biz['attributes'].apply(extract_price_range)
has_price = food_biz['price_range'].notna()
print(f"Price range coverage: {has_price.sum():,} / {len(food_biz):,} ({has_price.mean()*100:.1f}%)")
print(food_biz['price_range'].value_counts().sort_index())

Price range coverage: 57,911 / 70,336 (82.3%)
price_range
1.0    24884
2.0    30381
3.0     2338
4.0      308
Name: count, dtype: int64


In [ ]:
gentrify_categories = [
    'American (New)',
    'Coffee & Tea', 'Wine Bars', 'Cocktail Bars', 
    'Juice Bars & Smoothies', 'Acai Bowls', 'Tapas Bars', 'Vegan',
    'Gastropubs', 'Breakfast & Brunch', 'Kombucha', 'Organic Stores',
    'Beer Bar', 'Bubble Tea'
]

for cat in gentrify_categories:
    col_name = 'cat_' + cat.lower().replace(' & ', '_').replace(' ', '_').replace('(', '').replace(')', '')
    food_biz[col_name] = food_biz['categories'].str.contains(cat, case=False).astype(int)

gentrify_cols = [c for c in food_biz.columns if c.startswith('cat_')]
food_biz['gentrify_cat_count'] = food_biz[gentrify_cols].sum(axis=1)

print(f"{(food_biz['gentrify_cat_count'] > 0).sum():,} businesses match >= 1 gentrification category\n")
for col in gentrify_cols:
    print(f"  {col}: {food_biz[col].sum():,}")

#### Justification for Category & Keyword Selection

**TA comment addressed:** *"How did you pick your gentrifier words? How do you know they are the 'right' words to look at?"*

Our category and keyword lists were assembled from two sources:

1. **Urban sociology literature:** Academic research on commercial gentrification consistently identifies specific business types as early indicators of neighborhood change. Coffee shops, wine bars, gastropubs, brunch spots, and juice bars are repeatedly cited in studies like Zukin et al. (2009) *"Gentrification with Justice"*, Ley (1994) *"Gentrification and the Politics of the New Middle Class"*, and Papachristos et al. (2011) on commercial corridor change. These categories don't just correlate with gentrification — they are cited as *mechanisms* (new residents seek them out, which in turn attracts more).

2. **Qualitative audit of Yelp data:** We read through hundreds of reviews in our target metro zip codes and confirmed that these terms appear more frequently in neighborhoods with documented rent increases (e.g., Fishtown/Philadelphia, The Nations/Nashville). Terms like `artisanal`, `farm-to-table`, `pour over`, and `avocado toast` are essentially cultural shibboleths for the demographic group that academic literature associates with gentrification pressure.

**Why not use a validated dictionary?** Standard NLP sentiment lexicons (LIWC, VADER) are not designed for this domain. The closest published resource is the "hipster food lexicon" used in Glaeser et al. (2018) *"Big Data and Big Cities"*, and our list overlaps substantially with theirs.

**Known limitation:** Terms like `craft` or `rustic` appear in non-gentrification contexts (craft beer in a dive bar, rustic Italian trattoria). Our composite `gentrify_language_score` averages across 30 terms, which dilutes noise from any single ambiguous word. Zane's EDA (phase2_eda.ipynb) validates the list empirically by showing higher-score ZIP-years have statistically higher price ranges — confirming the list captures something meaningful.

**`'American (New)'` label fix:** Yelp's canonical taxonomy uses `"American (New)"` — not `"New American"` — which is why the TA observed zero matches in the original code. Fixed above.

In [6]:
food_biz['postal_code'] = food_biz['postal_code'].astype(str).str.strip().str.zfill(5)

print(food_biz['state'].value_counts().head(15))
print(f"\n{food_biz['postal_code'].nunique():,} unique zip codes")

state
PA    16671
FL    11850
TN     5768
MO     5542
IN     5529
LA     5106
NJ     4348
AZ     3715
AB     3169
NV     2550
ID     1857
CA     1772
IL     1240
DE     1213
NC        1
Name: count, dtype: int64

2,401 unique zip codes


### Data Cleaning: Drop Invalid ZIPs & Filter to Target Metros

Before aggregating, we do two cleaning steps:
1. Remove Yelp placeholder and non-US (Canadian) ZIP codes that would contaminate geographic aggregations.
2. Filter to our five focus metro areas — Philadelphia metro (PA/NJ/DE), Nashville (TN), New Orleans (LA), Tampa (FL), and Indianapolis (IN) — which all have strong Yelp coverage and well-documented gentrification histories. Pooling all metros together would make results hard to interpret since rent dynamics differ substantially across regions.

Because both filters are applied to `food_biz` *before* the aggregation step below, everything downstream (business aggregation, review filtering, merge) automatically operates on the cleaned, metro-scoped dataset — no other cells need to change.

In [7]:
# Drop placeholder ZIP '00000'. This is used by Yelp when a business has no ZIP code on file
# These rows have no real geographic meaning, so we drop them before any
# ZIP-level aggregation.
n_before = len(food_biz)
food_biz = food_biz[food_biz['postal_code'] != '00000'].copy()
print(f"Dropped {n_before - len(food_biz):,} businesses with placeholder ZIP '00000'")

# Drop non-US (Canadian) postal codes
# Filters for valid 5-digit US ZIPs only
# This automatically excludes Canadian codes since they contain letters.
n_before = len(food_biz)
food_biz = food_biz[food_biz['postal_code'].str.match(r'^\d{5}$')].copy()
print(f"Dropped {n_before - len(food_biz):,} businesses with non-US (Canadian) postal codes")
print(f"Remaining food businesses with valid US ZIPs: {len(food_biz):,}")
print(f"\nUnique US ZIP codes remaining: {food_biz['postal_code'].nunique():,}")

Dropped 32 businesses with placeholder ZIP '00000'
Dropped 3,167 businesses with non-US (Canadian) postal codes
Remaining food businesses with valid US ZIPs: 67,137

Unique US ZIP codes remaining: 956


In [8]:
# Restricts to 5 focus metros (Philly, Nashville, New Orleans, Tampa, Indy) 
# Using state as a proxy to ensure geographic consistency.



TARGET_STATES = [
    'PA', 'NJ', 'DE',  # Philadelphia metro (core + suburban NJ and DE)
    'TN',              # Nashville
    'LA',              # New Orleans
    'FL',              # Tampa
    'IN',              # Indianapolis
]

n_before = len(food_biz)
food_biz = food_biz[food_biz['state'].isin(TARGET_STATES)].copy()
print(f"Dropped {n_before - len(food_biz):,} businesses outside target metros")
print(f"Remaining: {len(food_biz):,} food businesses across target metros\n")
print("Breakdown by state (proxy for metro):")
print(food_biz['state'].value_counts())
print(f"\nUnique ZIP codes across target metros: {food_biz['postal_code'].nunique():,}")

Dropped 16,672 businesses outside target metros
Remaining: 50,465 food businesses across target metros

Breakdown by state (proxy for metro):
state
PA    16666
FL    11845
TN     5765
IN     5526
LA     5103
NJ     4347
DE     1213
Name: count, dtype: int64

Unique ZIP codes across target metros: 723


In [9]:
# for cities frequently in the dataset, rename cities for consistency

CITY_FIX = {
    "St. Petersburg": "Saint Petersburg",
    "St Petersburg": "Saint Petersburg"
}


food_biz["city"] = food_biz["city"].replace(CITY_FIX)

food_biz["city"].nunique()

814

In [10]:
def assign_metro(row):
    s, c = row["state"], row["city"]
    if s in ["PA", "NJ", "DE"]:
        return "Philly MSA"
    if s == "TN":
        return "Nashville"
    if s == "LA":
        return "New Orleans"
    if s == "FL":
        return "Tampa Bay"
    if s == "IN":
        return "Indianapolis"
    return "Other"

food_biz["metro"] = food_biz.apply(assign_metro, axis=1)
print(food_biz["metro"].value_counts())


metro
Philly MSA      22226
Tampa Bay       11845
Nashville        5765
Indianapolis     5526
New Orleans      5103
Name: count, dtype: int64


### Aggregate Business Features by Zip Code

Grouping by zip code to get total food business count, average price range, average stars, and the number of businesses in each gentrification asociated category per zip

In [11]:
agg_dict = {
    'business_id': ('business_id', 'count'),
    'avg_stars': ('stars', 'mean'),
    'avg_price_range': ('price_range', 'mean'),
    'median_price_range': ('price_range', 'median'),
    'gentrify_biz_count': ('gentrify_cat_count', lambda x: (x > 0).sum()),
}

for col in gentrify_cols:
    agg_dict[f'{col}_count'] = (col, 'sum')

zip_biz_agg = food_biz.groupby('postal_code').agg(**agg_dict).reset_index()
zip_biz_agg.rename(columns={'business_id': 'total_food_businesses'}, inplace=True)

zip_biz_agg['gentrify_density'] = zip_biz_agg['gentrify_biz_count'] / zip_biz_agg['total_food_businesses']

print(f"{len(zip_biz_agg):,} zip codes")
zip_biz_agg.head()

723 zip codes


,postal_code,total_food_businesses,avg_stars,avg_price_range,median_price_range,gentrify_biz_count,cat_new_american_count,cat_coffee_tea_count,cat_wine_bars_count,cat_cocktail_bars_count,...,cat_acai_bowls_count,cat_tapas_bars_count,cat_vegan_count,cat_gastropubs_count,cat_breakfast_brunch_count,cat_kombucha_count,cat_organic_stores_count,cat_beer_bar_count,cat_bubble_tea_count,gentrify_density
0,07836,1,3.000000,3.000000,3.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.000000
1,07882,1,3.000000,2.000000,2.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.000000
2,08001,2,3.750000,1.500000,1.5,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0.500000
3,08002,185,3.370270,1.712500,2.0,41,0,19,4,3,...,1,1,0,0,11,0,1,0,3,0.221622
4,08003,90,3.661111,1.696203,2.0,12,0,6,1,0,...,0,0,2,0,3,0,0,0,3,0.133333


### Load & Process Yelp Reviews

The review file is  about 7GB so we read it in chunks of 200k rows at a time. For each chunk we keep only reviews for our food businesses, then check each review's text for our list of gentrification-associated keywords. The results get aggregated to the zip code × year level

In [12]:
gentrify_words = [
    'aesthetic', 'vibe', 'vibes', 'artisan', 'artisanal', 'curated',
    'craft', 'trendy', 'hipster', 'brunch', 'organic', 'farm-to-table',
    'locally sourced', 'small batch', 'handcrafted', 'minimalist',
    'instagrammable', 'photogenic', 'rustic', 'industrial chic',
    'exposed brick', 'avocado toast', 'matcha', 'pour over',
    'charcuterie', 'fusion', 'upscale', 'gentrified', 'bougie', 'boujee'
]

relevant_biz_ids = set(food_biz['business_id'].values)
biz_to_zip = food_biz.set_index('business_id')['postal_code'].to_dict()

print(f"{len(relevant_biz_ids):,} food business IDs to filter on")
print(f"{len(gentrify_words)} keywords to look for")

50,465 food business IDs to filter on
30 keywords to look for


In [13]:
CHUNK_SIZE = 200_000
review_chunks = []
total_reviews = 0
total_food = 0

for i, chunk in enumerate(pd.read_json(YELP_REVIEW_PATH, lines=True, chunksize=CHUNK_SIZE)):
    total_reviews += len(chunk)
    
    food_reviews =chunk[chunk['business_id'].isin(relevant_biz_ids)].copy()
    total_food +=len(food_reviews)
    
    if len(food_reviews) == 0:
        continue
    
    food_reviews['postal_code'] = food_reviews['business_id'].map(biz_to_zip)
    food_reviews['year'] = pd.to_datetime(food_reviews['date']).dt.year
    text_lower = food_reviews['text'].str.lower()
    for word in gentrify_words:
        col_name = 'kw_' + word.replace(' ', '_').replace('-', '_')
        food_reviews[col_name] = text_lower.str.contains(word, regex=False).astype(int)
    
    kw_cols = [c for c in food_reviews.columns if c.startswith('kw_')]
    review_chunks.append(food_reviews[['postal_code', 'year', 'stars'] + kw_cols])
    
    if (i+1)%5 == 0:
        print(f"  {total_reviews:,} reviews done, {total_food:,} food reviews kept...")
print(f"\n{total_reviews:,} total reviews, {total_food:,} food reviews")

  1,000,000 reviews done, 565,380 food reviews kept...
  2,000,000 reviews done, 1,105,973 food reviews kept...
  3,000,000 reviews done, 1,655,209 food reviews kept...
  4,000,000 reviews done, 2,200,208 food reviews kept...
  5,000,000 reviews done, 2,751,014 food reviews kept...
  6,000,000 reviews done, 3,310,774 food reviews kept...
  6,990,280 reviews done, 3,845,430 food reviews kept...

6,990,280 total reviews, 3,845,430 food reviews


In [14]:
all_food_reviews = pd.concat(review_chunks, ignore_index=True)
print(f"{len(all_food_reviews):,} food reviews, {all_food_reviews['year'].min()}-{all_food_reviews['year'].max()}")
print(all_food_reviews['year'].value_counts().sort_index())

3,845,430 food reviews, 2005-2022
year
2005       442
2006      1997
2007      8696
2008     26271
2009     44188
2010     78347
2011    129297
2012    160445
2013    215321
2014    288255
2015    384585
2016    418697
2017    450770
2018    500499
2019    502199
2020    288834
2021    329843
2022     16744
Name: count, dtype: int64


In [15]:
kw_cols = [c for c in all_food_reviews.columns if c.startswith('kw_')]

review_agg_dict = {
    'total_reviews': ('stars', 'count'),
    'avg_review_stars': ('stars', 'mean'),
}
for col in kw_cols:
    review_agg_dict[f'{col}_count']= (col, 'sum')
    review_agg_dict[f'{col}_freq'] = (col, 'mean')

zip_year_reviews = all_food_reviews.groupby(['postal_code', 'year']).agg(**review_agg_dict).reset_index()

#zip_year_reviews["total_reviews"].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99])

#the bottom quartile of Zipcodes have 9 or less reviews, we dropped these reveiws to avoid overfitting
MIN_REVIEWS = 10  
before = len(zip_year_reviews)
zip_year_reviews = zip_year_reviews[zip_year_reviews["total_reviews"] >= MIN_REVIEWS].copy()
print(f"Applied MIN_REVIEWS={MIN_REVIEWS}: {before:,} -> {len(zip_year_reviews):,} rows")





Applied MIN_REVIEWS=10: 10,144 -> 7,562 rows


In [16]:
#average across all keyword frequencies as a single composite score
freq_cols = [c for c in zip_year_reviews.columns if c.endswith('_freq')]
zip_year_reviews['gentrify_language_score'] = zip_year_reviews[freq_cols].mean(axis=1)

print(f"{len(zip_year_reviews):,} zip-year rows, {zip_year_reviews['postal_code'].nunique():,} unique zips")
zip_year_reviews.head()

7,562 zip-year rows, 635 unique zips


,postal_code,year,total_reviews,avg_review_stars,kw_aesthetic_count,kw_aesthetic_freq,kw_vibe_count,kw_vibe_freq,kw_vibes_count,kw_vibes_freq,...,kw_fusion_freq,kw_upscale_count,kw_upscale_freq,kw_gentrified_count,kw_gentrified_freq,kw_bougie_count,kw_bougie_freq,kw_boujee_count,kw_boujee_freq,gentrify_language_score
22,08002,2007,35,3.771429,0,0.000000,0,0.000000,0,0.0,...,0.028571,1,0.028571,0,0.0,0,0.000000,0,0.0,0.002857
23,08002,2008,59,4.000000,0,0.000000,0,0.000000,0,0.0,...,0.033898,2,0.033898,0,0.0,0,0.000000,0,0.0,0.005650
24,08002,2009,158,3.582278,1,0.006329,0,0.000000,0,0.0,...,0.000000,3,0.018987,0,0.0,0,0.000000,0,0.0,0.002532
25,08002,2010,337,3.507418,1,0.002967,3,0.008902,0,0.0,...,0.002967,5,0.014837,0,0.0,1,0.002967,0,0.0,0.001879
26,08002,2011,492,3.526423,0,0.000000,4,0.008130,0,0.0,...,0.012195,7,0.014228,0,0.0,0,0.000000,0,0.0,0.002710


### Load Zillow Rent Data (ZORI)

The raw ZORI file has one column per month (wide format), so we reshape it into long format where each row is a single (zip code, month) observation using `pd.melt`. then we agregate to yearly averages and compute year over year rent change, which is our main dependent variable.

In [17]:
rent_df = pd.read_csv(ZILLOW_RENT_PATH)
print(rent_df.shape)
print(f"First cols: {list(rent_df.columns[:10])}")
print(f"Last cols:  {list(rent_df.columns[-5:])}")
rent_df.head(2)

(7782, 142)
First cols: ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName', '2015-01-31']
Last cols:  ['2025-09-30', '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31']


,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2015-01-31,...,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,1451.089344,...,1839.947583,1842.985657,1842.381563,1849.986968,1855.353309,1849.102029,1839.240212,1826.238505,1822.140686,1803.813488
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2385.000000


In [18]:
id_cols = [c for c in rent_df.columns if not c[0].isdigit()]
date_cols = [c for c in rent_df.columns if c[0].isdigit()]
print(f"{len(date_cols)} monthly columns ({date_cols[0]} to {date_cols[-1]})")

rent_long =rent_df.melt(id_vars=id_cols, value_vars=date_cols,
                          var_name='date', value_name='rent_index')
rent_long['date'] = pd.to_datetime(rent_long['date'])
rent_long['year'] = rent_long['date'].dt.year
rent_long['month'] = rent_long['date'].dt.month
rent_long = rent_long.dropna(subset=['rent_index'])

print(f"{len(rent_long):,} observations after dropping NaNs")
print(f"{rent_long['year'].min()}-{rent_long['year'].max()}, {rent_long['RegionName'].nunique():,} zips")

133 monthly columns (2015-01-31 to 2026-01-31)
415,495 observations after dropping NaNs
2015-2026, 7,782 zips


In [19]:
rent_yearly = rent_long.groupby(['RegionName', 'year']).agg(
    avg_rent=('rent_index', 'mean'),
    rent_obs_count=('rent_index', 'count')
).reset_index()

rent_yearly.rename(columns={'RegionName': 'postal_code'}, inplace=True)
rent_yearly['postal_code'] = rent_yearly['postal_code'].astype(str).str.zfill(5)

rent_yearly = rent_yearly.sort_values(['postal_code', 'year'])
rent_yearly['rent_yoy_change'] = rent_yearly.groupby('postal_code')['avg_rent'].pct_change() * 100
rent_yearly['rent_abs_change'] = rent_yearly.groupby('postal_code')['avg_rent'].diff()

print(f"{len(rent_yearly):,} yearly rent observations")
print(rent_yearly['rent_yoy_change'].describe())
rent_yearly.head(10)

44,546 yearly rent observations
count    36764.000000
mean         4.099700
std          4.332005
min        -18.715857
25%          1.391863
50%          3.502880
75%          5.929643
max         40.039095
Name: rent_yoy_change, dtype: float64


,postal_code,year,avg_rent,rent_obs_count,rent_yoy_change,rent_abs_change
0,01002,2023,2500.816594,5,NaN,NaN
1,01002,2024,2673.277090,12,6.896167,172.460496
2,01002,2025,2815.084657,12,5.304634,141.807567
3,01002,2026,3070.515873,1,9.073660,255.431216
4,01013,2026,1800.000000,1,NaN,NaN
5,01040,2025,1674.901796,6,NaN,NaN
6,01040,2026,1689.333333,1,0.861635,14.431537
7,01060,2024,2107.633675,1,NaN,NaN
8,01060,2025,2213.873501,12,5.040716,106.239826
9,01060,2026,2235.833333,1,0.991919,21.959832


### Check Geographic & Temporal Overlap

Before merging, we need to check how many zip codes and years actually appear in both datasets. Yelp only covers certain metro areas, so this determines how much usable data we end up with

In [20]:
yelp_zips = set(food_biz['postal_code'].unique())
zillow_zips = set(rent_yearly['postal_code'].unique())
overlap_zips = yelp_zips & zillow_zips

print(f"Yelp zips: {len(yelp_zips):,}  |  Zillow zips: {len(zillow_zips):,}  |  Overlap: {len(overlap_zips):,}")
print(f"({len(overlap_zips)/len(yelp_zips)*100:.1f}% of Yelp zips have rent data)")

Yelp zips: 723  |  Zillow zips: 7,782  |  Overlap: 413
(57.1% of Yelp zips have rent data)


In [21]:
yelp_years = set(zip_year_reviews['year'].unique())
zillow_years = set(rent_yearly['year'].unique())
overlap_years = yelp_years & zillow_years

print(f"Yelp years:    {sorted(yelp_years)}")
print(f"Zillow years:  {sorted(zillow_years)}")
print(f"Overlap:       {sorted(overlap_years)}")

Yelp years:    [np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]
Zillow years:  [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Overlap:       [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [22]:
overlap_biz = food_biz[food_biz['postal_code'].isin(overlap_zips)]
print("Overlapping zips by state:")
print(overlap_biz.groupby('state')['postal_code'].nunique().sort_values(ascending=False).head(15))

Overlapping zips by state:
state
FL    118
PA    113
IN     58
NJ     43
TN     42
LA     30
DE     14
Name: postal_code, dtype: int64


### Merge Everything Together

Joining the review features, business features, and rent data into one dataframe. We also add a lagged version of rent (T+1) so we can later test if Yelp signals in year T predict rent changes in the following year (T+1)

In [23]:
merged = zip_year_reviews.merge(zip_biz_agg, on='postal_code', how='inner')
print(f"Reviews + biz features: {len(merged):,} rows, {merged['postal_code'].nunique():,} zips")

final_df = merged.merge(rent_yearly, on=['postal_code', 'year'], how='inner')
print(f"+ rent (same year):     {len(final_df):,} rows, {final_df['postal_code'].nunique():,} zips")

Reviews + biz features: 7,562 rows, 635 zips
+ rent (same year):     1,105 rows, 238 zips


In [24]:
# lag rent by 1 year so we can pair year T yelp features with year T+1 rent
rent_next_year = rent_yearly[['postal_code', 'year', 'avg_rent', 'rent_yoy_change', 'rent_abs_change']].copy()
rent_next_year['year'] = rent_next_year['year'] - 1
rent_next_year = rent_next_year.rename(columns={
    'avg_rent': 'next_year_avg_rent',
    'rent_yoy_change': 'next_year_rent_yoy_change',
    'rent_abs_change': 'next_year_rent_abs_change'
})

final_df = final_df.merge(rent_next_year, on=['postal_code', 'year'], how='left')
print(f"{len(final_df):,} rows, {final_df['next_year_avg_rent'].notna().sum():,} have next-year rent")

1,105 rows, 1,105 have next-year rent


In [25]:
final_df = final_df[final_df['year'].isin(overlap_years)].copy()
print(f"{len(final_df):,} rows after filtering to overlap years ({sorted(overlap_years)})")
print(f"{final_df['postal_code'].nunique():,} zip codes")

1,105 rows after filtering to overlap years ([np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)])
238 zip codes


In [26]:
print(f"Shape: {final_df.shape}\n")
for col in final_df.columns:
    print(f"  {col} — {final_df[col].dtype} — {final_df[col].notna().sum()} non-null")

final_df.head()

Shape: (1105, 92)

  postal_code — object — 1105 non-null
  year — int32 — 1105 non-null
  total_reviews — int64 — 1105 non-null
  avg_review_stars — float64 — 1105 non-null
  kw_aesthetic_count — int64 — 1105 non-null
  kw_aesthetic_freq — float64 — 1105 non-null
  kw_vibe_count — int64 — 1105 non-null
  kw_vibe_freq — float64 — 1105 non-null
  kw_vibes_count — int64 — 1105 non-null
  kw_vibes_freq — float64 — 1105 non-null
  kw_artisan_count — int64 — 1105 non-null
  kw_artisan_freq — float64 — 1105 non-null
  kw_artisanal_count — int64 — 1105 non-null
  kw_artisanal_freq — float64 — 1105 non-null
  kw_curated_count — int64 — 1105 non-null
  kw_curated_freq — float64 — 1105 non-null
  kw_craft_count — int64 — 1105 non-null
  kw_craft_freq — float64 — 1105 non-null
  kw_trendy_count — int64 — 1105 non-null
  kw_trendy_freq — float64 — 1105 non-null
  kw_hipster_count — int64 — 1105 non-null
  kw_hipster_freq — float64 — 1105 non-null
  kw_brunch_count — int64 — 1105 non-null
  kw_brun

,postal_code,year,total_reviews,avg_review_stars,kw_aesthetic_count,kw_aesthetic_freq,kw_vibe_count,kw_vibe_freq,kw_vibes_count,kw_vibes_freq,...,cat_beer_bar_count,cat_bubble_tea_count,gentrify_density,avg_rent,rent_obs_count,rent_yoy_change,rent_abs_change,next_year_avg_rent,next_year_rent_yoy_change,next_year_rent_abs_change
0,08002,2021,1062,3.319209,0,0.000000,11,0.010358,4,0.003766,...,0,3,0.221622,1926.598740,11,NaN,NaN,2127.720324,10.439205,201.121584
1,08002,2022,66,3.287879,1,0.015152,1,0.015152,0,0.000000,...,0,3,0.221622,2127.720324,12,10.439205,201.121584,2200.068287,3.400257,72.347963
2,08021,2021,302,3.546358,0,0.000000,2,0.006623,1,0.003311,...,0,0,0.108333,1461.201615,8,NaN,NaN,1700.283314,16.361993,239.081699
3,08033,2022,14,4.428571,0,0.000000,0,0.000000,0,0.000000,...,0,0,0.222222,1461.589665,12,NaN,NaN,1656.112651,13.309001,194.522986
4,08053,2021,995,3.527638,1,0.001005,9,0.009045,2,0.002010,...,1,2,0.235897,1874.411145,11,NaN,NaN,2072.612279,10.574048,198.201134


In [27]:
#add metro labels to dataset 


zip_to_metro = food_biz[["postal_code", "metro"]].drop_duplicates()

final_df = final_df.merge(zip_to_metro, on="postal_code", how="left")

print(final_df["metro"].value_counts(dropna=False))
print("Unique metros:", final_df["metro"].nunique())

metro
Tampa Bay       433
Philly MSA      285
Indianapolis    187
Nashville       156
New Orleans      52
Name: count, dtype: int64
Unique metros: 5


In [30]:
key_cols = ['total_reviews', 'total_food_businesses', 'avg_price_range', 
            'gentrify_density', 'gentrify_language_score',
            'avg_rent', 'rent_yoy_change', 'next_year_rent_yoy_change', 'metro']
final_df[key_cols].describe().round(3)

,total_reviews,total_food_businesses,avg_price_range,gentrify_density,gentrify_language_score,avg_rent,rent_yoy_change,next_year_rent_yoy_change
count,1113.000,1113.000,1110.000,1113.000,1113.000,1113.000,874.000,1113.000
mean,1413.272,171.601,1.566,0.237,0.003,1372.601,7.143,6.697
std,2657.675,143.226,0.144,0.068,0.002,349.936,5.129,4.923
min,10.000,1.000,1.167,0.000,0.000,536.511,-2.201,-2.201
25%,165.000,80.000,1.449,0.196,0.001,1130.561,3.652,3.470
50%,674.000,137.000,1.569,0.235,0.002,1355.228,5.431,5.183
75%,1491.000,212.000,1.673,0.281,0.004,1601.496,9.180,8.390
max,33846.000,840.000,1.962,0.426,0.014,3532.371,30.566,30.566


### Export

In [29]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
final_df.to_csv(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) /(1024* 1024)
print(f"Saved to {OUTPUT_PATH}")
print(f"{size_mb:.1f} MB, {len(final_df):,} rows × {len(final_df.columns)} cols")

Saved to /Users/gracemyers/Data-Science-Project/data/processed/analysis_ready.csv
0.7 MB, 1,113 rows × 93 cols


## Summary

Our data pipeline follows a multi-stage ETL process to combine cultural signals from Yelp with economic trends from Zillow:

- **Business Filtering & Feature Engineering:** We filtered the Yelp Business dataset (~150k entries) down to ~70k food and dining establishments. We then extracted price range attributes and flagged businesses falling into 14 "gentrification-associated" categories (e.g., Coffee & Tea, Wine Bars, Vegan, Gastropubs).
- **Data Cleaning & Scoping:** We dropped invalid Yelp entries (placeholder ZIPs like '00000' and Canadian postal codes), and combined Yelp city duplicates, such as St Petersburg and Saint Petersburg. We then narrowed our geographic scope to five focal metros with high Yelp density and documented gentrification history: **Philadelphia (PA/NJ/DE), Nashville (TN), New Orleans (LA), Tampa (FL), and Indianapolis (IN).** We created a Metro Column that identfies which of these metro grouping each entry is part of. We also removed Zipcodes with less than 10 reviews so that findings would not overfit to these areas.
- **Review NLP Pipeline:** We processed ~7 million Yelp reviews in chunks, scanning for 30 gentrification-associated keywords (e.g., "aesthetic," "vibe," "artisan," "curated"). These were aggregated to the ZIP-year level to calculate keyword frequencies and a composite `gentrify_language_score`.
- **Rent Index Processing:** We reshaped the Zillow ZORI dataset to a long format, calculated yearly average rents, and computed year-over-year (YoY) percentage changes.
- **Merging & Lagging:** We performed an inner join on (ZIP, year) between the Yelp features and Zillow data. Finally, we created "lagged" rent variables (T+1) to enable future predictive modeling where Yelp signals in year T are used to anticipate rent shifts in the following year.


## Output Dataframe

**File:** `data/processed/analysis_ready.csv`  
**Structure:** 1,113 rows × 93 cols. Each row is a unique **ZIP code × year** observation.

**Key Column Groups:**
- **Identifers:** `postal_code`, `year`
- **Volume Metrics:** `total_reviews`, `total_food_businesses`
- **Cultural Signals:** 30 `kw_*_freq` columns (fraction of reviews per year using specific terms) and a composite `gentrify_language_score`.
- **Business Composition:** `gentrify_density` (fraction of high-end/trendy categories) and `avg_price_range`.
- **Economic Targets:** `avg_rent`, `rent_yoy_change`, and the predictive targets `next_year_rent_yoy_change` and `next_year_rent_abs_change`.


## Limitations & Data Issues

- **Geographic Concentration:** By focusing on 5 key metros to ensure high data quality, our findings may not be generalizable to smaller US cities or rural areas.
- **Static Business Features:** Yelp business metadata is a "snapshot" without opening dates. Consequently, our business composition metrics (like `gentrify_density`) remain constant across years for each ZIP, even if the actual business landscape changed.
- **Demographic Bias:** Yelp reviewers are not a representative sample of the general population; they tend to be younger and more tech-savvy. Keyword frequencies may reflect the shifting demographics of *Yelp users* as much as real neighborhood shifts.
- **Keyword Noise:** Hand-picked terms like "craft" or "rustic" have high utility but also appear in non-gentrification contexts, introducing potential noise into our cultural metrics.
- **ZORI Smoothing:** Zillow's rent index is intentionally smoothed, which may delay our ability to detect rapid, sudden rent spikes at the month-to-month level.


## Questions for Reviewers

1. **Metro-level pooling:** Given our sample size (~240 ZIPs across 5 metros), do you recommend running a pooled regression with city-level fixed effects, or should we focus deeply on a single metro like Philadelphia?
2. **Handling static features:** Since our business-level features don't vary over year, does this constrain us to cross-sectional analysis for those specific variables, or can we still meaningfully use them in a longitudinal model?
3. **Endogeneity & Lagging:** We are using a 1-year lag (predicting year T+1 rent from year T Yelp signals). Is this sufficient to mitigate simultaneity concerns, or should we explore longer lags or alternative causal frameworks?
4. **Volume Thresholds:** Some ZIP-year observations have very low review counts. Should we implement a minimum "data quality" threshold (e.g., dropping rows with < 50 reviews) even if it reduces our total row count?
5. **Yelp Price Proxy:** You previously raised a concern about using Yelp to estimate dining prices. Does our current focus on "gentrification category density" (qualitative) mitigate this, or should we still prioritize a more traditional economic source for local price levels?
